In [1]:
!pip install -q langgraph langchain-google-genai

In [2]:
# ============================================================
# MULTI-AGENT CUSTOMER SUPPORT SYSTEM
# Using LangGraph
# ============================================================

# Install LangGraph
!pip install -q langgraph


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


# ============================================================
# 2. DEFINE STATE
# ============================================================

class AgentState(TypedDict):

    messages: Annotated[
        list[BaseMessage],
        add_messages
    ]

    next_node: str


# ============================================================
# 3. REFUND TOOL
# ============================================================

def process_refund(user_id: str, amount: float) -> str:

    return (
        f"SUCCESS: Refund of ${amount:.2f} has been "
        f"processed for User '{user_id}'."
    )


# ============================================================
# 4. SUPERVISOR AGENT
# ============================================================

def supervisor_agent(state: AgentState) -> AgentState:

    # HumanMessage object -> use .content
    user_query = state["messages"][-1].content.lower()

    # Decide which agent should handle the request
    if (
        "refund" in user_query
        or "billed" in user_query
        or "billing" in user_query
        or "account" in user_query
    ):

        next_step = "account_actions_agent"

    else:

        next_step = "tech_support_agent"

    return {
        "next_node": next_step
    }


# ============================================================
# 5. TECHNICAL SUPPORT AGENT
# ============================================================

def tech_support_agent(state: AgentState) -> AgentState:

    user_query = state["messages"][-1].content

    response = (
        "[Tech Support]\n\n"
        "I understand that your app is freezing when "
        "you upload a PNG file.\n\n"
        "Try these troubleshooting steps:\n"
        "1. Check the PNG file size.\n"
        "2. Try uploading another PNG file.\n"
        "3. Restart the application.\n"
        "4. Clear the application's cache.\n"
        "5. Check file permissions.\n"
        "6. Update the application.\n"
        "7. If the issue continues, reinstall the application."
    )

    return {
        "messages": [
            AIMessage(content=response)
        ],
        "next_node": END
    }


# ============================================================
# 6. ACCOUNT ACTION AGENT
# ============================================================

def account_actions_agent(state: AgentState) -> AgentState:

    user_query = state["messages"][-1].content.lower()

    # Default values for demonstration
    user_id = "user_9876"
    amount = 49.99

    # Try to extract amount from the query
    import re

    amount_match = re.search(
        r"\$?\s*(\d+(?:\.\d+)?)",
        user_query
    )

    if amount_match:
        amount = float(amount_match.group(1))

    # Process refund
    refund_result = process_refund(
        user_id,
        amount
    )

    response = (
        "[Account Agent]\n\n"
        "Account action requested.\n"
        f"Tool Result: {refund_result}"
    )

    return {
        "messages": [
            AIMessage(content=response)
        ],
        "next_node": END
    }


# ============================================================
# 7. CREATE LANGGRAPH WORKFLOW
# ============================================================

workflow = StateGraph(AgentState)


# Add nodes
workflow.add_node(
    "supervisor",
    supervisor_agent
)

workflow.add_node(
    "tech_support_agent",
    tech_support_agent
)

workflow.add_node(
    "account_actions_agent",
    account_actions_agent
)


# ============================================================
# 8. CONNECT THE NODES
# ============================================================

# START -> Supervisor

workflow.add_edge(
    START,
    "supervisor"
)


# Supervisor -> Appropriate Agent

workflow.add_conditional_edges(

    "supervisor",

    lambda state: state["next_node"],

    {
        "tech_support_agent":
            "tech_support_agent",

        "account_actions_agent":
            "account_actions_agent",

        END:
            END
    }
)


# Technical Support -> END

workflow.add_edge(
    "tech_support_agent",
    END
)


# Account Actions -> END

workflow.add_edge(
    "account_actions_agent",
    END
)


# ============================================================
# 9. COMPILE THE GRAPH
# ============================================================

app = workflow.compile()


# ============================================================
# 10. RUN DEMO FUNCTION
# ============================================================

def run_demo(user_query: str):

    print("\n========================================")
    print("USER QUERY")
    print("========================================")

    print(user_query)

    inputs = {
        "messages": [
            HumanMessage(content=user_query)
        ]
    }

    result = app.invoke(inputs)

    print("\n========================================")
    print("SYSTEM RESPONSE")
    print("========================================")

    print(
        result["messages"][-1].content
    )


# ============================================================
# 11. TEST CASE 1
# ============================================================

run_demo(
    "My app keeps freezing whenever I try to "
    "upload a PNG file. How can I fix this?"
)


# ============================================================
# 12. TEST CASE 2
# ============================================================

run_demo(
    "I was billed twice by mistake. "
    "Please refund $49.99 for my account user_9876."
)


USER QUERY
My app keeps freezing whenever I try to upload a PNG file. How can I fix this?

SYSTEM RESPONSE
[Tech Support]

I understand that your app is freezing when you upload a PNG file.

Try these troubleshooting steps:
1. Check the PNG file size.
2. Try uploading another PNG file.
3. Restart the application.
4. Clear the application's cache.
5. Check file permissions.
6. Update the application.
7. If the issue continues, reinstall the application.

USER QUERY
I was billed twice by mistake. Please refund $49.99 for my account user_9876.

SYSTEM RESPONSE
[Account Agent]

Account action requested.
Tool Result: SUCCESS: Refund of $49.99 has been processed for User 'user_9876'.
